# Cleanup: 04 — Dataplex scans and profiling descriptions

**Removes** artifacts from `04_gdelt_data_profiling.ipynb`:

- **Dataplex Data Profile** scans: IDs follow `profile-{dataset}-{table}` (underscores → hyphens, lowercased, max 63 chars).
- **Dataplex Data Documentation** scans: IDs follow `docs-{dataset}-{table}` with the same normalization.
- **BigQuery** table and column descriptions applied by that notebook are **cleared** (set to empty).

**BigQuery dataset missing:** If `{BIGQUERY_DATASET}` was already removed (e.g. after `01` cleanup), listing tables and clearing descriptions are skipped with no error. Dataplex scan deletion still runs **best-effort** for **`BIGQUERY_TABLES`** in `config.py` only.

**IAM**: Dataplex Administrator or equivalent to delete DataScans; BigQuery permissions to update table metadata.

**Irreversible** for AI-generated descriptions unless you restore them manually.

**Config:** `{BIGQUERY_DATASET}` matches **`BIGQUERY_DATASET`** from **`config.py`** (loaded by the code cells below).

### Full reset order

1. **`02_gdelt_graph_create_cleanup.ipynb`** — Drop the BigQuery property graph while datasets still exist.
2. **`03_gdelt_incremental_refresh_cleanup.ipynb`** — Remove `_sync_metadata` and staging tables (optional if you immediately run step 4).
3. **`04_gdelt_data_profiling_cleanup.ipynb`** — Remove Dataplex scans and generated descriptions (BigQuery steps are skipped if the dataset no longer exists).
4. **`01_gdelt_data_prep_cleanup.ipynb`** — Delete `{BIGQUERY_DATASET}`, `{BIGQUERY_DATASET}_us`, and local exports (**always last**).

```mermaid
flowchart LR
  cleanup02[cleanup_02_drop_graph]
  cleanup03[cleanup_03_incremental_artifacts]
  cleanup04[cleanup_04_dataplex_and_descriptions]
  cleanup01[cleanup_01_datasets_and_local]
  cleanup02 --> cleanup01
  cleanup03 --> cleanup01
  cleanup04 --> cleanup01
```

In [ ]:
import sys
from pathlib import Path

_cwd = Path.cwd().resolve()
_cfg_dir = _cwd.parent if _cwd.name == "clean_up" else _cwd
if not (_cfg_dir / "config.py").is_file():
    _cfg_dir = _cwd
sys.path.insert(0, str(_cfg_dir))

from config import (
    GCP_PROJECT_ID,
    PROJECT_REGION,
    BIGQUERY_DATASET,
    BIGQUERY_TABLES,
    print_config,
    validate_config,
)

print_config()
if not validate_config():
    print("\n⚠️  Update config.py before proceeding.")
else:
    print("\n✅ Configuration loaded.")

In [ ]:
from google.api_core.exceptions import NotFound as ApiNotFound
from google.cloud import bigquery
from google.cloud import dataplex_v1
from google.cloud.exceptions import NotFound as CloudNotFound


def dataplex_scan_id(prefix: str, dataset_id: str, table_id: str) -> str:
    """Match notebook 04 normalization rules."""
    return (
        f"{prefix}-{dataset_id}-{table_id}".replace("_", "-").lower()[:63].rstrip("-")
    )


def schema_without_descriptions(fields):
    """Recursively rebuild schema with descriptions cleared."""
    out = []
    for field in fields:
        nested = ()
        if field.fields:
            nested = tuple(schema_without_descriptions(field.fields))
        out.append(
            bigquery.SchemaField(
                name=field.name,
                field_type=field.field_type,
                mode=field.mode,
                description=None,
                fields=nested,
                policy_tags=field.policy_tags,
                precision=field.precision,
                scale=field.scale,
                max_length=field.max_length,
                range_element_type=field.range_element_type,
                rounding_mode=getattr(field, "rounding_mode", None),
                foreign_type_definition=getattr(field, "foreign_type_definition", None),
                default_value_expression=getattr(field, "default_value_expression", None),
            )
        )
    return out


bq_client = bigquery.Client(project=GCP_PROJECT_ID)
dataplex_client = dataplex_v1.DataScanServiceClient()
parent = f"projects/{GCP_PROJECT_ID}/locations/{PROJECT_REGION}"

dataset_ref = f"{GCP_PROJECT_ID}.{BIGQUERY_DATASET}"
tables = None
try:
    tables = list(bq_client.list_tables(dataset_ref))
except CloudNotFound:
    print(
        f"ℹ️  Dataset `{GCP_PROJECT_ID}.{BIGQUERY_DATASET}` not found — "
        "skipping BigQuery description cleanup.\n"
        "   Best-effort Dataplex deletes use BIGQUERY_TABLES from config only "
        "(other scans may remain)."
    )

if tables is None:
    table_names = list(BIGQUERY_TABLES)
elif tables:
    table_names = [t.table_id for t in tables]
else:
    table_names = []

print(f"Tables to clean (Dataplex): {len(table_names)}")

for table_id in table_names:
    for prefix in ("profile", "docs"):
        scan_id = dataplex_scan_id(prefix, BIGQUERY_DATASET, table_id)
        name = f"{parent}/dataScans/{scan_id}"
        try:
            operation = dataplex_client.delete_data_scan(name=name)
            operation.result(timeout=300)
            print(f"✅ Deleted Dataplex scan {scan_id}")
        except (ApiNotFound, CloudNotFound):
            print(f"⏭️  No Dataplex scan {scan_id}")
        except Exception as e:
            print(f"⚠️  {scan_id}: {e}")

if tables is not None and tables:
    for table_id in table_names:
        table_ref = f"{GCP_PROJECT_ID}.{BIGQUERY_DATASET}.{table_id}"
        table = bq_client.get_table(table_ref)
        table.description = None
        table.schema = schema_without_descriptions(table.schema)
        bq_client.update_table(table, ["description", "schema"])
        print(f"✅ Cleared descriptions for {table_id}")
elif tables is not None and not tables:
    print("ℹ️  Dataset exists but has no tables — skipping BigQuery description cleanup.")

print("\n✅ Profiling cleanup finished.")